In [1]:
from pyspark import SparkContext, SparkConf

conf = SparkConf().setAppName("Lab2").setMaster("local[*]")
sc = SparkContext(conf=conf)

raw_rdd = sc.textFile("data/Reviews.csv") # Suppose root where you called `uv run jupyter lab` was DistrArch/

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 21:21:32 WARN Utils: Your hostname, cubone, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlan0)
26/04/22 21:21:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 21:21:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
header = raw_rdd.first()

In [3]:
print(header)

Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text


In [5]:
# 1. Ignore first line (header)
# 2. Fetch UserId, ProductId in a PairRDD-fashion (key is UserID)
# 3. This will trigger a shuffle, so much data will move around the network
# 4. Then we prepare data to be lists, so that later the sum is a concatenation
# 5. Sum of lists is a concatenation
# 6. Return the values only
aggregated_rdd = raw_rdd.filter(lambda line: line.strip() != header.strip()) \
    .map(lambda line: (line.split(',')[2], line.split(',')[1])) \
    .distinct() \
    .map(lambda x: (x[0], [x[1]])) \
    .reduceByKey(lambda listProd1, listProd2: listProd1 + listProd2) \
    .values()

# 7. Apply same procedure as in hadoop: double windowing + sorting (so that A, B == B, A)
all_combinations_rdd = aggregated_rdd.flatMap(
    lambda prods: [tuple(sorted([prods[i], prods[j]])) for i in range(len(prods)-1) for j in range(i+1, len(prods))]
)

In [6]:
# OR (below is more efficient)

In [7]:
# 1. Ignore first line (header)
# 2. Fetch UserId, ProductId in a PairRDD-fashion (key is UserID)
# 3. This will trigger a shuffle, so much data will move around the network
# 4. Then we prepare data to be lists, so that later the sum is a concatenation
# 5. Sum of lists is a concatenation
# 6. Return the values only
# EXTRA!!: WE SORT THE PRODUCTS HERE INSTEAD OF DOING IT IN PLACE IN THE flatMap() AT STEP 7
aggregated_rdd_v2 = raw_rdd.filter(lambda line: line.strip() != header.strip()) \
    .map(lambda line: (line.split(',')[2], line.split(',')[1])) \
    .distinct() \
    .map(lambda x: (x[0], [x[1]])) \
    .reduceByKey(lambda listProd1, listProd2: listProd1 + listProd2) \
    .values() \
    .map(lambda prods: sorted(prods))

# 7. Apply same procedure as in hadoop: double windowing + sorting (so that A, B == B, A)
all_combinations_rdd_v2 = aggregated_rdd_v2.flatMap(
    lambda prods: [(prods[i], prods[j]) for i in range(len(prods)-1) for j in range(i+1, len(prods))]
)

In [8]:
# Count frequency and sort by such frequency from larger to smaller
frequency_combinations_rdd = all_combinations_rdd_v2.map(lambda prodPair: (prodPair, 1)).reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[1], ascending=False)

In [9]:
frequency_combinations_rdd.saveAsTextFile("output/result")

In [10]:
sc.stop

<bound method SparkContext.stop of <SparkContext master=local[*] appName=Lab2>>